# SETUP

In [7]:
!pip install chromadb==0.4.18 pypdf

In [8]:
!pip install langchain-text-splitters

# CONFGIURE

In [9]:
document_content = """
# Dummy Driving Test Document

## Section 1: Road Signs
- Stop signs are red and octagonal.
- Yield signs are red and inverted triangles.
- Speed limit signs are white and rectangular.

## Section 2: Traffic Laws
- Always stop at red lights.
- Yield to pedestrians in crosswalks.
- Do not text and drive.

## Section 3: Parking Rules
- Do not park within 15 feet of a fire hydrant.
- Parallel parking requires signaling and checking mirrors.
"""
with open("driving_test_document.md", "w") as f:
    f.write(document_content)

document_path = "driving_test_document.md"
print(f"Dummy document saved to {document_path}")

Dummy document saved to driving_test_document.md


In [10]:
!pip install langchain-community langchain langchain_classic langchain-google-genai
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_classic.chains import RetrievalQA
import os
from google.colab import userdata

# Ensure the API key is configured for all operations
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Load the document
loader = TextLoader(document_path)
documents = loader.load()

# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)
print(f"Split {len(documents)} document into {len(splits)} chunks.")

Split 1 document into 1 chunks.


In [11]:
# Imports for all necessary components
import os
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

# --- Document content and path definition (from cell 527c930f) ---
document_content = """
# Dummy Driving Test Document

## Section 1: Road Signs
- Stop signs are red and octagonal.
- Yield signs are red and inverted triangles.
- Speed limit signs are white and rectangular.

## Section 2: Traffic Laws
- Always stop at red lights.
- Yield to pedestrians in crosswalks.
- Do not text and drive.

## Section 3: Parking Rules
- Do not park within 15 feet of a fire hydrant.
- Parallel parking requires signaling and checking mirrors.
"""
with open("driving_test_document.md", "w") as f:
    f.write(document_content)
document_path = "driving_test_document.md"
print(f"Dummy document saved to {document_path}")

# --- API key setup, document loading, and text splitting (from cell 399d3040) ---
# Ensure the API key is configured
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Load the document
loader = TextLoader(document_path)
documents = loader.load()

# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)
print(f"Split {len(documents)} document into {len(splits)} chunks.")

# --- Embeddings initialization, numpy install, and vector store creation (from original a1aa7030) ---
# Initialize Google Generative AI Embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", api_key=os.environ["GOOGLE_API_KEY"])

# To resolve numpy compatibility issue, downgrade numpy if necessary.
# This command will be re-executed if the environment requires it.
!pip install numpy==1.26.4

# Create Persistent Chroma vector store
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings, persist_directory="./chroma_db")
print("Chroma vector store created and persisted to ./chroma_db")

Dummy document saved to driving_test_document.md
Split 1 document into 1 chunks.


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Chroma vector store created and persisted to ./chroma_db


In [12]:
import google.generativeai as genai
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Available models that support embedContent:")
for m in genai.list_models():
  if "embedContent" in m.supported_generation_methods:
    print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available models that support embedContent:
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [17]:
# Initialize the Gemini LLM with gemini-2.5-flash
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

# Create a retrieval chain, specifying k=1 as there is only 1 chunk in the vector store
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectorstore.as_retriever(search_kwargs={"k": 1}))
print("Retrieval QA chain created.")

Retrieval QA chain created.


# TESTING USER INPUT

In [18]:
# Ask a question based on the document
query = "What are the rules about parking near a fire hydrant?"
response = qa_chain.invoke({"query": query})
print("\nQuery:", query)
print("Response:", response["result"])


Query: What are the rules about parking near a fire hydrant?
Response: Do not park within 15 feet of a fire hydrant.


In [19]:
# Ask out of context
query = "What is the capital of France?"
response = qa_chain.invoke({"query": query})
print("\nQuery:", query)
print("Response:", response["result"])


Query: What is the capital of France?
Response: I don't know the answer to that question based on the provided document.
